In [1]:
import pandas as pd
import numpy as np
import json
import glob

import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

import spacy
from nltk.corpus import stopwords
import pyLDAvis
import pyLDAvis.gensim

In [2]:
def load_data(file):
    with open (file, "r", encoding="utf-8") as f:
        data = json.load(f)
    return (data)

def write_data(file, data):
    with open (file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [3]:
stopwords = stopwords.words("english")
stopwords.append("be")
print(stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [4]:
data = load_data("vir_transcripts.json")["Transcripts"]
print (data[0][0:90])
print (data[1][0:90])

I lost 80% of my mind. It is very freeing. You should see the look on your faces right now
Yeah, that was a great transition. I went from an English medium school to a school where 


In [5]:
def lemmatization(texts, allowed_postages=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    texts_out = []
    for text in data:
        doc = nlp(text)
        new_text = []
        for token in doc:
            if token.pos_ in allowed_postages:
                new_text.append(token.lemma_)
        final = " ".join(new_text)
        texts_out.append(final)
    return (texts_out)

lemmatized_texts = lemmatization(data)
print (lemmatized_texts[0][0:90])

lose % mind very freeing should see look face right now way good evening guy excited all r


In [6]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download()


showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

In [7]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use', 'be', 'know', 'go', 'think', 'see', 'guy', 'say', 'even', 'year', 'one', 'would', 'find'])
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/alisha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
def gen_words(data):
    final = []
    for text in data:
        new = gensim.utils.simple_preprocess(text, deacc=True)
        final.append(new)
    return (final)

data_words = gen_words(lemmatized_texts)

print (data_words[0][0:20])
print (data_words[1][0:20])

['lose', 'mind', 'very', 'freeing', 'should', 'see', 'look', 'face', 'right', 'now', 'way', 'good', 'evening', 'guy', 'excited', 'all', 'right', 'name', 'be', 'go']
['great', 'transition', 'go', 'english', 'medium', 'school', 'school', 'where', 'speak', 'english', 'medium', 'go', 'noun', 'pronoun', 'verb', 'chest', 'shoulder', 'tricep', 'go', 'could']


In [9]:
data_words = remove_stopwords(data_words)
print(data_words[:1][0][:30])

['lose', 'mind', 'freeing', 'look', 'face', 'right', 'way', 'good', 'evening', 'excited', 'right', 'name', 'good', 'time', 'tonight', 'excited', 'delightful', 'talk', 'time', 'really', 'embrace', 'root', 'make', 'comedy', 'authentically', 'indian', 'really', 'could', 'indian', 'fake']


In [10]:
#BIGRAMS AND TRIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=200)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=100)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return([bigram[doc] for doc in texts])

def make_trigrams(texts):
    return ([trigram[bigram[doc]] for doc in texts])

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams)

print (data_bigrams_trigrams[0])

['lose', 'mind', 'freeing', 'look', 'face', 'right', 'way', 'good', 'evening', 'excited', 'right', 'name', 'good', 'time', 'tonight', 'excited', 'delightful', 'talk', 'time', 'really', 'embrace', 'root', 'make', 'comedy', 'authentically', 'indian', 'really', 'could', 'indian', 'fake', 'american', 'accent', 'understand', 'opportunity', 'make', 'history', 'tonight', 'first', 'ever', 'come', 'leave', 'never', 'happen', 'stick', 'around', 'kick', 'news', 'week', 'work', 'well', 'leave', 'pasture', 'honestly', 'honestly', 'government', 'ban', 'beef', 'international', 'career', 'may', 'bad', 'thing', 'make', 'mistake', 'beef', 'good', 'couple', 'first', 'world', 'tour', 'entire', 'world', 'like', 'country', 'world', 'common', 'like', 'thing', 'number', 'masturbate', 'country', 'thank', 'chain', 'dna', 'everywhere', 'hotel', 'memory', 'foam', 'mattress', 'memory', 'matter', 'entire', 'world', 'people', 'thing', 'indian', 'love', 'indian', 'people', 'smart', 'indian', 'people', 'smart', 'lead'

In [11]:
# TF-IDF REMOVAL
from gensim.models import TfidfModel

id2word = corpora.Dictionary(data_bigrams_trigrams)

texts = data_bigrams_trigrams

corpus = [id2word.doc2bow(text) for text in texts]
print (corpus[0][0:20])

tfidf = TfidfModel(corpus, id2word=id2word)

low_value = 0.03
words = []
words_missing_in_tfidf = []
for i in range (0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] # reinitialization to be safe, you can skip this
    tfidf_ids = [id for id, value in bow]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words+words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # the words with tf-idf score 0 will be missing
    
    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]
    corpus[i] = new_bow
    
    



[(0, 1), (1, 1), (2, 1), (3, 2), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 2), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 3)]


In [12]:
# id2word = corpora.Dictionary(data_words)

# corpus = []
# for text in data_words:
#     new = id2word.doc2bow(text)
#     corpus.append(new)
    
# print (corpus[0][0:20])

# word = id2word[[0][:1][0]]
# print(word)

In [13]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, 
                                           id2word=id2word,
                                           num_topics=30,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

In [14]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=30)
vis

TypeError: drop() takes from 1 to 2 positional arguments but 3 were given

In [8]:
!which python

/Library/Frameworks/Python.framework/Versions/3.8/bin/python
